# 03 Feature Selection with LASSO and RFECV

This notebook compares two feature selection approaches:

- LASSO Logistic Regression
- Recursive Feature Elimination with Cross-Validation (RFECV)

The goal is to identify whether a smaller subset of features can achieve similar predictive performance.

## Imports and Preprocessing

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import GridSearchCV
from sklearn.feature_selection import RFECV
from sklearn.ensemble import RandomForestClassifier

try:
    from xgboost import XGBClassifier
except ImportError:
    print("Please install xgboost: pip install xgboost")

RANDOM_STATE = 42



# Update these paths if your CSV files are stored elsewhere.
train_path = "../data/train.csv"
test_path = "../data/test.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Remove unnamed index columns if present
train_df = train_df.loc[:, ~train_df.columns.str.contains("^Unnamed")]
test_df = test_df.loc[:, ~test_df.columns.str.contains("^Unnamed")]

# Impute missing arrival delay values using the training median
arrival_delay_median = train_df["Arrival Delay in Minutes"].median()
train_df["Arrival Delay in Minutes"] = train_df["Arrival Delay in Minutes"].fillna(arrival_delay_median)
test_df["Arrival Delay in Minutes"] = test_df["Arrival Delay in Minutes"].fillna(arrival_delay_median)

# Encode categorical variables
nominal_cols = ["Gender", "Type of Travel"]
ordinal_cols = ["Customer Type", "Class"]

ohe = OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)
oe = OrdinalEncoder(
    categories=[
        ["disloyal Customer", "Loyal Customer"],
        ["Eco", "Eco Plus", "Business"]
    ]
)
le = LabelEncoder()

train_nominal = pd.DataFrame(
    ohe.fit_transform(train_df[nominal_cols]),
    columns=ohe.get_feature_names_out(nominal_cols),
    index=train_df.index
)

test_nominal = pd.DataFrame(
    ohe.transform(test_df[nominal_cols]),
    columns=ohe.get_feature_names_out(nominal_cols),
    index=test_df.index
)

train_ordinal = pd.DataFrame(
    oe.fit_transform(train_df[ordinal_cols]),
    columns=ordinal_cols,
    index=train_df.index
)

test_ordinal = pd.DataFrame(
    oe.transform(test_df[ordinal_cols]),
    columns=ordinal_cols,
    index=test_df.index
)

train_encoded = train_df.drop(columns=nominal_cols + ordinal_cols).join(train_nominal).join(train_ordinal)
test_encoded = test_df.drop(columns=nominal_cols + ordinal_cols).join(test_nominal).join(test_ordinal)

train_encoded["satisfaction"] = le.fit_transform(train_encoded["satisfaction"])
test_encoded["satisfaction"] = le.transform(test_encoded["satisfaction"])

# Separate features and target
X_train = train_encoded.drop(columns=["satisfaction", "id"])
y_train = train_encoded["satisfaction"]

X_test = test_encoded.drop(columns=["satisfaction", "id"])
y_test = test_encoded["satisfaction"]

# Scale features for models that benefit from standardization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training data shape:", X_train.shape)
print("Test data shape:", X_test.shape)
print("Target classes:", list(le.classes_))

## LASSO Logistic Regression

In [ ]:
lasso = LogisticRegressionCV(
    Cs=10,
    cv=5,
    penalty="l1",
    solver="liblinear",
    max_iter=1000
)

lasso.fit(X_train_scaled, y_train)
lasso_pred = lasso.predict(X_test_scaled)

lasso_results = {
    "Model": "LASSO Logistic Regression",
    "Best C": lasso.C_[0],
    "Accuracy": accuracy_score(y_test, lasso_pred),
    "Precision": precision_score(y_test, lasso_pred),
    "Recall": recall_score(y_test, lasso_pred),
    "F1 Score": f1_score(y_test, lasso_pred)
}

selected_features_lasso = X_train.columns[(lasso.coef_ != 0).ravel()]

print(lasso_results)
print("\nSelected features after LASSO:")
print(list(selected_features_lasso))

## RFECV Feature Selection

In [ ]:
rfecv = RFECV(
    estimator=LogisticRegression(max_iter=1000),
    step=1,
    cv=5,
    scoring="accuracy"
)

rfecv.fit(X_train_scaled, y_train)
rfecv_pred = rfecv.predict(X_test_scaled)

rfecv_results = {
    "Model": "RFECV Logistic Regression",
    "Selected Features": rfecv.n_features_,
    "Accuracy": accuracy_score(y_test, rfecv_pred),
    "Precision": precision_score(y_test, rfecv_pred),
    "Recall": recall_score(y_test, rfecv_pred),
    "F1 Score": f1_score(y_test, rfecv_pred)
}

selected_features_rfecv = X_train.columns[rfecv.support_]

print(rfecv_results)
print("\nSelected features after RFECV:")
print(list(selected_features_rfecv))

## Save Feature Selection Results

In [ ]:
feature_selection_results = pd.DataFrame([lasso_results, rfecv_results])
feature_selection_results.to_csv("../results/feature_selection_results.csv", index=False)

pd.DataFrame({"LASSO_Selected_Features": selected_features_lasso}).to_csv(
    "../results/lasso_selected_features.csv", index=False
)

pd.DataFrame({"RFECV_Selected_Features": selected_features_rfecv}).to_csv(
    "../results/rfecv_selected_features.csv", index=False
)

print("Feature selection results saved.")